# bridge_board

> Phase 20 (`specs/multiagent/phase20/`, experiment C1): squads with coupled work and bridges. `BridgeBoardEnv` is `ColonyBoardEnv` in squads where every fourth subtask a squad receives is coupled to the next squad around a ring: its clue 0 is dealt in squad s and its cross clue (clue 1) in squad s + 1, with no new random draws. The holders of the cross clue in s + 1 see the subtask as a mirror: they may post the clue there (`mirrorLog`, never the channel) but cannot open or answer it. Bridges (`dealBridges`: one or two honest non-guard slots per squad, or its first guard) read their pair's mirror posts and relay a value onto their own squad's subtask (`relayLog`, `relaySource`); under the credit schemes a relayed clue's credit is split between the bridge and the first true poster on the mirror. `BridgeView` is one agent's view. Built only when a bridge key is set; every earlier board is unchanged.

In [ ]:
#| default_exp bridge_board

In [ ]:
#| hide
from nbdev.showdoc import *

## Bridge slots

In [ ]:
#| export
from explRL.colony_board import ColonyBoardEnv, SquadView, dealSquads

BRIDGE_KEYS = ('couple', 'bridges', 'bridgeGuard')


def dealBridges(N, S, nLiar, guards, perPair=1, guarding=False):
    """[[bridge slots] per squad], squads placed as squadOf (slot i in squad i mod (N // S)); liars the first nLiar slots and
    guards the next `guards` (as dealSquads). Ordinary: the first perPair honest non-guard slots of each squad, by slot order;
    guarding: the first guard slot of each squad (perPair must be 1). ValueError if a squad lacks the slots."""
    if perPair not in (0, 1, 2) or guarding and perPair != 1:
        raise ValueError(f'bridges: perPair must be 0, 1 or 2 (1 when guarding), got {perPair!r}')
    n, (_, gs) = N // S, dealSquads(N, S, nLiar, guards)
    out = [(gs[s] if guarding else [i for i in range(nLiar + guards, N) if i % n == s])[:perPair] for s in range(n)]
    if any(len(b) < perPair for b in out):
        raise ValueError(f"bridges: a squad has fewer than {perPair} {'guard' if guarding else 'honest non-guard'} slots")
    return out


def checkBridge(world, N=None, nLiar=0, zeroOk=False):
    "The bridge keys' checks: couple needs squads, bridges couple, bridgeGuard bridges = 1 and guards; dealBridges must succeed."
    if not any(k in world for k in BRIDGE_KEYS):
        return True
    S, b = world.get('squads'), world.get('bridges', 0)
    c = world.get('couple')
    if 'couple' in world and not (zeroOk and c == 0) and not (S and isinstance(c, (int, float)) and 0 < c <= 1 and abs(1 / c - round(1 / c)) < 1e-9):
        raise ValueError('couple needs squads and a value 1/n in (0, 1] (every n-th subtask)')
    if 'couple' not in world or b and not world.get('couple') or world.get('bridgeGuard') and (b != 1 or not world.get('guards', 0)):
        raise ValueError('a bridge key needs couple (no silent default); bridges need couple > 0; bridgeGuard needs bridges = 1 and guards')
    if N is not None and S:
        if N // S < 2:
            raise ValueError('couple needs at least two squads')
        dealBridges(N, S, nLiar, world.get('guards', 0), b, bool(world.get('bridgeGuard')))
    return True

## The board

In [ ]:
#| export
class BridgeBoardEnv(ColonyBoardEnv):
    """ColonyBoardEnv in squads (S) with coupled subtasks and bridges. couple (0.25): each squad's every 1/couple-th assigned
    subtask is coupled to squad (s + 1) mod n: clue 0 dealt within s, clue 1 within s + 1 (two holders each, _pick on rngDeal).
    Mirrors: squad s + 1 sees each coupled subtask of s as a mirror; its agents' shares there go to mirrorLog. bridges (0, 1, 2
    per pair) and bridgeGuard: bridge slots from dealBridges; a bridge's share on a coupled subtask of a value in that subtask's
    mirrorLog is a relay (relayLog). Every other behaviour is ColonyBoardEnv's."""
    def __init__(self, seed, nAgents, couple=0.25, bridges=0, bridgeGuard=False, nLiar=0, **kw):
        self.couple, self.nLiar, S = float(couple or 0), int(nLiar), kw.get('squads')
        g = len(frozenset(kw.get('guardSlots', ())))
        checkBridge(dict(couple=self.couple, bridges=bridges, bridgeGuard=bridgeGuard, squads=S, guards=g), nAgents, self.nLiar, zeroOk=True)
        if frozenset(kw.get('guardSlots', ())) != frozenset(range(self.nLiar, self.nLiar + g)):
            raise ValueError('bridges: guard slots must follow the nLiar liar slots')
        self.bridges = dealBridges(int(nAgents), int(S), self.nLiar, g, int(bridges), bool(bridgeGuard)) if S else []
        self.bridgeSet, self.bridgeGuard = frozenset(a for b in self.bridges for a in b), bool(bridgeGuard)
        super().__init__(seed, nAgents, **kw)

    def reset(self, seed, nAgents):
        "mirrorLog, relayLog, coupled {k: (s, s2)} and nAssigned per squad, then the base reset (which deals 2N + 2 subtasks)."
        self.nAssigned, self.coupled, self.mirrorLog, self.relayLog = [0] * self.nSquads, {}, [], []
        super().reset(seed, nAgents)

    def _deal(self):
        "Once per subtask, from ClueBoardEnv._new (self._squad and self.serial set): every 1/couple-th of a squad's is coupled."
        s = self._squad
        self.nAssigned[s] += 1
        if not self.couple or self.nAssigned[s] % round(1 / self.couple):
            return super()._deal()
        s2 = self.coupled.setdefault(self.serial, (s, (s + 1) % self.nSquads))[1]
        slots = lambda q: [a for a in range(self.nAgents) if a % self.nSquads == q]
        held, out = {a: 0 for a in slots(s) + slots(s2)}, []
        for j in range(self.m):                       # as ColonyBoardEnv._deal, clue 1 from s2's slots, the others from s's
            sl, hs = slots(s2 if j == 1 else s), set()
            groups = ([[a for a in sl if a in self.guardSet], [a for a in sl if a not in self.guardSet]]
                      if self.guardDeal else [sl] * min(self.r, len(sl)))
            for g in groups:
                a = self._pick([a for a in g if a not in hs], held)
                hs.add(a)
                held[a] += 1
            out.append(frozenset(hs))
        return out

    def mirrors(self, i, every=None):
        """The open coupled subtasks k whose s2 is i's squad and whose clue 1 i holds (P20-16); every (default: i is a liar
        slot, P20-5): all of them."""
        q, every = i % self.nSquads, i < self.nLiar if every is None else every
        return [k for k in self.open if k in self.coupled and self.coupled[k][1] == q
                and (every or i in self.tasks[k].holders[1])]

    def mirrorTally(self, k):
        "tally's format, {k: {j: [(r, senders, newest)]}} ({} with no posts), from mirrorLog: mirror posts never reach the channel."
        vals, out = {}, {}
        for t, a, k2, j, r, *_ in self.mirrorLog:
            if k2 == k:
                s, n = vals.get((j, r), (set(), -1.0))
                vals[(j, r)] = (s | {a}, max(n, t))
        for (j, r), (s, n) in vals.items():
            out.setdefault(k, {}).setdefault(j, []).append((r, frozenset(s), n))
        for j in out.get(k, {}):
            out[k][j].sort(key=lambda x: (-len(x[1]), -x[2], x[0]))
        return out

    def bridgeOf(self, i):
        "(s, s2) if agent i is a bridge (from s2 = s + 1 to its squad s), else None."
        s = i % self.nSquads
        return (s, (s + 1) % self.nSquads) if i in self.bridgeSet else None

    def apply(self, i, action, t):
        """A share of clue 1 on one of mirrors(i) goes to mirrorLog (a 'shared' ledger row, counts['shares']; not shareLog);
        a bridge's share on its pair's coupled k of a value in k's mirrorLog is also logged in relayLog with its source post
        (the value's first mirror post). Opening or answering a mirror stays invalid (ColonyBoardEnv: another squad's subtask)."""
        kind, arg = action
        if kind == 'share' and int(arg[0]) in self.coupled:
            self.advance(t)                                    # mirrors(i) lists the subtasks open at t
            k, j, r = (int(x) for x in arg)
            if j == 1 and k in self.mirrors(i):
                red = any(m[2] == k and m[3] == j for m in self.mirrorLog)
                self.mirrorLog.append((float(t), i, k, j, r, red, self.tasks[k].task.clue(j)[1] != r))
                self.counts['shares'] += 1
                rew, dur, _, info = super().apply(i, ('wait', None), t)   # the turn as a wait (time, contest view, reward) ...
                self.ledger[-1] = self.ledger[-1][:4] + (k,) + self.ledger[-1][5:]   # ... its ledger row on k
                return rew, dur, k, dict(info, task=k, result='shared')
            relay = self.coupled[k] == self.bridgeOf(i)                     # a bridge's share on its pair's subtask
            src = next((m for m in self.mirrorLog if m[2:5] == (k, j, r)), None) if relay else None
            n, out = len(self.shareLog), super().apply(i, action, t)
            if src is not None and len(self.shareLog) > n:
                self.relayLog.append((float(t), i, k, j, r, src))
            return out
        return super().apply(i, action, t)

    def _creditTo(self, i, k):
        """[(agent, d)], each paid credit * delta / d: the base agents with d = their number; under 'credit' and 'contest' a relay's
        share goes half to the bridge and half to its source poster, the first true poster on the mirror (P20-12)."""
        out = []
        for j, hs in enumerate(self.tasks[k].holders):
            if i not in hs:
                s = next((s for s in self.shareLog if s[2] == k and s[3] == j and not s[6] and s[1] != i), None)
                if s is not None and s[1] not in [x[0] for x in out]:
                    src = relaySource(self, s)
                    out.append([s[1]] + ([src[1]] if src is not None else []))
        return [(a, len(out) * len(to)) for to in out for a in to]

    def _reward(self, i, delta, k=None, solved=False):
        if not (self.scheme in ('credit', 'contest') and solved and (to := self._creditTo(i, k))):
            return super()._reward(i, delta, k, solved)
        for a, d in to:
            self.pending[a] += self.credit * delta / d
        self.pending[i] += (1 - self.credit) * delta
        r, self.pending[i] = float(self.pending[i]), 0.0
        return r

    def viewFor(self, i):
        return BridgeView(self, i)


class BridgeView(SquadView):
    "SquadView plus `mirrors` (agent i's mirrors, with the cross clues it holds) and, for a bridge, `pair` (mirror tallies)."
    def __init__(self, board, i):
        super().__init__(board, i)
        self.__dict__['_i'] = i

    def mirrors(self, i=None, every=None):
        "The board's mirrors(i): agent i's mirror subtasks (holders of the cross clue; liars: all)."
        return self._b.mirrors(self._i if i is None else i, every)

    def crossClues(self):
        "{k: (p, r)}: the cross clue agent i holds on each of its mirrors."
        return {k: self._b.tasks[k].task.clue(1) for k in self.mirrors() if self._i in self._b.tasks[k].holders[1]}

    @property
    def pair(self):
        "For a bridge, tally's format over its pair's open coupled subtasks with mirror posts; None for anyone else."
        p, out = self._b.bridgeOf(self._i), {}
        if p is None:
            return None
        for k in self._b.open:
            if self._b.coupled.get(k) == p:
                out.update(self._b.mirrorTally(k))
        return out


def relaySource(board, post):
    "The mirrorLog post a relay copied (post: its shareLog or relayLog entry, (t, i, k, j, r, ...)), or None."
    return next((x[5] for x in getattr(board, 'relayLog', ()) if x[:5] == tuple(post[:5])), None)

## Tests

In [ ]:
# test_couple (AC-2, FR-A1, P20-1): every fourth subtask per squad is coupled, clue 0 held in s and clue 1 in s + 1 (two
# holders each), never both clues to one agent; values and clues equal the `squads` board's on the same seed
from explRL.colony_board import squadOf
from explRL.colony_grid import WORLD18

kw, G = dict(WORLD18), range(16, 32)                       # 64 agents in squads of 8: liars 0-15, guards 16-31
for seed in (1800, 1801, 1890):
    a = ColonyBoardEnv(seed, 64, guardSlots=G, squads=8, **kw)
    b = BridgeBoardEnv(seed, 64, guardSlots=G, squads=8, nLiar=16, couple=0.25, bridges=1, **kw)
    for t in range(1, 61):                                  # replacements deal too
        a.advance(t), b.advance(t)
    assert a.serial == b.serial > 300 and a.squadOfTask == b.squadOfTask
    assert all(a.tasks[k].value == b.tasks[k].value and a.tasks[k].task == b.tasks[k].task for k in a.tasks)
    for s in range(8):
        ks = sorted(k for k in b.tasks if b.squadOfTask[k] == s)
        assert [k in b.coupled for k in ks] == [n % 4 == 3 for n in range(len(ks))] and b.nAssigned[s] == len(ks)
    first = min(b.coupled)                                  # the deal matches the squads board's up to the first coupled subtask
    assert all(a.tasks[k].holders == b.tasks[k].holders for k in range(first))
    for k, h in b.tasks.items():
        s = b.squadOfTask[k]
        assert not (h.holders[0] & h.holders[1]) and all(len(hs) == 2 for hs in h.holders)
        if k in b.coupled:
            assert b.coupled[k] == (s, (s + 1) % 8)
            assert {squadOf(x, 64, 8) for x in h.holders[0]} == {s} and {squadOf(x, 64, 8) for x in h.holders[1]} == {(s + 1) % 8}
        else:
            assert all(squadOf(x, 64, 8) == s for hs in h.holders for x in hs)
    assert abs(len(b.coupled) / len(b.tasks) - 0.25) < 0.02

# without coupling (couple 0, no bridges) the board is the squads board, deal included
z = BridgeBoardEnv(1800, 64, guardSlots=G, squads=8, nLiar=16, couple=0, **kw)
a = ColonyBoardEnv(1800, 64, guardSlots=G, squads=8, **kw)
assert all(z.tasks[k].holders == a.tasks[k].holders and z.tasks[k].task == a.tasks[k].task for k in a.tasks) and not z.coupled
# coupling with guardDeal keeps one guard copy per clue, in the clue's own squad
g = BridgeBoardEnv(3, 64, guardSlots=G, squads=8, guardDeal=True, nLiar=16, couple=0.25, **kw)
assert all(len(hs & set(G)) == 1 for k in g.coupled for hs in g.tasks[k].holders)

In [ ]:
# test_mirror_isolation (AC-3, FR-A1, FR-A3, P20-16), the board's half: a mirror share goes to mirrorLog, never to shareLog
# or the channel, so squad s's tally, colonyLine and knownColony never show it; mirrorTally reads mirrorLog; squad s + 1
# cannot open or answer a mirror; only holders of the cross clue (and liar slots) of s + 1 see it, other squads never.
# Left for T2: the prompt (TextClueEnv.observe with bridgeLines), parse, and numpy posters (bridgeMembers, MirrorLiar).
from explRL.clue_agents import ClueBoard
from explRL.clue_colony import colonyLine, knownColony, tally
from explRL.colony_board import squadView

b = BridgeBoardEnv(1800, 64, guardSlots=G, squads=8, nLiar=16, couple=0.25, bridges=1, **kw)
k = next(k for k in b.open if k in b.coupled and max(b.tasks[k].holders[0]) >= 16)   # with an honest clue-0 holder
s, s2 = b.coupled[k]
task, (p1, r1) = b.tasks[k], b.tasks[k].task.clue(1)
h1, h2 = sorted(task.holders[1])                            # the cross clue's holders, in s2
o = next(x for x in task.holders[0] if x >= 16)             # an honest holder of clue 0, in s
non = next(x for x in range(32, 64) if x % 8 == s2 and x not in task.holders[1])   # honest, in s2, no cross clue
liar = next(x for x in range(16) if x % 8 == s2)
far = next(x for x in range(32, 64) if x % 8 not in (s, s2))
assert k in b.mirrors(h1) and k in b.mirrors(h2) and k in b.mirrors(liar)        # P20-5: liars of s2 draw from all mirrors
assert k not in b.mirrors(non) and k not in b.mirrors(o) and k not in b.mirrors(far) and k in b.mirrors(non, every=True)
v, vn, vf, vo = (squadView(b, x) for x in (h1, non, far, o))
assert isinstance(v, BridgeView) and k in v.mirrors() and v.crossClues()[k] == (p1, r1) and k not in v.open
assert k not in vn.mirrors() and k not in vf.mirrors() and k in vo.open and v.pair is None
assert all(x[0] != k for x in v.observe(h1)['open'])

ch = ClueBoard()                                            # the blackboard: what TextClueEnv.post writes, from shares only
def act(i, a, t):
    n = len(b.shareLog)
    out = b.apply(i, a, t)
    if len(b.shareLog) > n:
        x = b.shareLog[-1]
        ch.post(i, x[2], (x[3], b.tasks[x[2]].task.primes[x[3]], x[4]), t)
    return out

c0 = dict(b.counts)
for i in (h1, h2):
    rew, _, loc, info = act(i, ('share', (k, 1, r1)), 1)
    assert loc == k and info['result'] == 'shared' and b.ledger[-1][4] == k and b.ledger[-1][0] == i
act(liar, ('share', (k, 1, (r1 + 1) % p1)), 2)
act(o, ('share', (k, 0, task.task.clue(0)[1])), 2)          # an ordinary share by squad s
assert b.counts['shares'] == c0['shares'] + 4 and len(b.shareLog) == 1 and len(b.mirrorLog) == 3
assert [m[1] for m in b.mirrorLog] == [h1, h2, liar] and [m[6] for m in b.mirrorLog] == [False, False, True]
assert b.mirrorLog[1][5] and not b.mirrorLog[0][5]          # redundant flag, as shareLog's
assert b.mirrorTally(k) == {k: {1: [(r1, frozenset({h1, h2}), 1.0), ((r1 + 1) % p1, frozenset({liar}), 2.0)]}}
assert b.mirrorTally(next(x for x in b.open if x != k)) == {}
tal = tally(ch, vo)
assert set(tal[k]) == {0} and 1 not in knownColony(o, vo, ch)[k]
assert f'x mod {p1} =' not in colonyLine(vo, tal) and f'x mod {task.task.primes[0]} =' in colonyLine(vo, tal)
assert all(k not in tally(ch, squadView(b, x)) for x in (h1, liar, far))
# invalid: opening a mirror (so answering it), a mirror share without the cross clue, clue 0 on a mirror, a mirror of another squad
for i, a in ((h1, ('open', k)), (non, ('share', (k, 1, r1))), (h1, ('share', (k, 0, 0))), (far, ('share', (k, 1, r1)))):
    try:
        b.apply(i, a, 3)
        raise AssertionError(f'{i} {a} should raise')
    except ValueError:
        pass
assert b.opened[h1] != k and len(b.mirrorLog) == 3 and len(b.shareLog) == 1
# an expired mirror is gone from mirrors(i)
b.advance(b.tasks[k].expire)
assert k not in b.mirrors(h1)

In [ ]:
# test_deal_bridges (AC-4, FR-A2, P20-2, AC-10's dealBridges part): one and two bridges per pair from the honest non-guard
# slots, guarding bridges from the guards, uneven liars (6 of 16 in squads of 4), clean boards, and the errors
assert dealBridges(64, 8, 16, 16) == [[32 + s] for s in range(8)]
assert dealBridges(64, 8, 16, 16, 2) == [[32 + s, 40 + s] for s in range(8)]
assert dealBridges(64, 8, 16, 16, 1, True) == [[16 + s] for s in range(8)]
assert dealBridges(64, 8, 16, 16, 0) == [[]] * 8
assert dealBridges(64, 8, 0, 16) == [[16 + s] for s in range(8)] and dealBridges(64, 8, 0, 16, 1, True) == [[s] for s in range(8)]
# 16 agents in squads of 4, 6 liars (2, 2, 1, 1) and 4 guards: squads 0 and 1 keep one honest non-guard slot
assert dealBridges(16, 4, 6, 4) == [[12], [13], [10], [11]] and dealBridges(16, 4, 6, 4, 1, True) == [[8], [9], [6], [7]]
assert dealBridges(16, 8, 4, 4) == [[8], [9]]
for args in ((16, 4, 6, 4, 2), (64, 8, 16, 16, 2, True), (64, 8, 16, 16, 3), (64, 8, 16, 0, 1, True), (64, 8, 16, 12),
             (16, 4, 8, 8, 1)):
    try:
        dealBridges(*args)
        raise AssertionError(f'{args} should raise')
    except ValueError:
        pass
# on the board: bridgeOf, and a guarding bridge is a guard; the team is unchanged (16 liars, 16 guards, 32 others)
for n, bg, want in ((1, False, [32 + s for s in range(8)]), (2, False, [32 + s for s in range(16)]), (1, True, list(G)[:8])):
    b = BridgeBoardEnv(1800, 64, guardSlots=G, squads=8, nLiar=16, couple=0.25, bridges=n, bridgeGuard=bg, **kw)
    assert sorted(b.bridgeSet) == want and all(b.bridgeOf(x) == (x % 8, (x + 1) % 8) for x in want)
    assert all(b.bridgeOf(x) is None for x in range(64) if x not in want) and (b.bridgeSet <= b.guardSet) == bg
# checkBridge (FR-B3's board checks)
w = dict(colony=True, guards=16, squads=8, couple=0.25, bridges=1)
assert checkBridge(w, 64, 16) and checkBridge(dict(w, bridgeGuard=True), 64, 16) and checkBridge({}, 64, 16)
for bad in (dict(w, squads=None), dict(w, couple=None), dict(w, bridges=2, bridgeGuard=True), dict(w, guards=0, bridgeGuard=True),
            dict(w, squads=64), dict(w, couple=1.5)):
    for f in (lambda: checkBridge(bad, 64, 16),
              lambda: BridgeBoardEnv(1, 64, guardSlots=range(16, 16 + bad['guards']), nLiar=16,
                                     **{k: bad[k] for k in ('squads', 'couple', 'bridges', 'bridgeGuard') if k in bad}, **kw)):
        try:
            f()
            raise AssertionError(f'{bad} should raise')
        except ValueError:
            pass

In [ ]:
# test_relay_credit (AC-6, FR-A5, P20-12; the board's half of AC-4's relay): a bridge's share of a mirror value is a relay
# logged with its source; relaySource finds the liar for a relayed lie; under 'credit' and 'contest' a relayed clue's credit
# is split equally between the bridge and the first true poster on the mirror; under 'individual' the rewards equal
# ColonyBoardEnv's for the same moves
from explRL.contest import contestOptions

def play(scheme, cls=BridgeBoardEnv):
    "Seed 1800, 16 agents in squads of 4 (4 liars, 4 guards, one bridge per pair); the moves on the first coupled subtask."
    x = dict(scheme=scheme, contest=contestOptions(dict(guards=4, contestSlots='all'), 16, 4) if scheme == 'contest' else None)
    b = BridgeBoardEnv(1800, 16, guardSlots=range(4, 8), squads=4, nLiar=4, couple=0.25, bridges=1, **kw, **x)
    k = next(k for k in b.open if k in b.coupled)
    s, s2 = b.coupled[k]
    task, br = b.tasks[k], b.bridges[s][0]
    (p1, r1), r0 = task.task.clue(1), task.task.clue(0)[1]
    h, liar = next(i for i in sorted(task.holders[1]) if i >= 4), next(i for i in range(4) if i % 4 == s2)
    solver = next(i for i in sorted(task.holders[0]) if i != br)
    lie = (r1 + 1) % p1
    mirror = [(liar, ('share', (k, 1, lie))), (h, ('share', (k, 1, r1)))]
    moves = [(br, ('share', (k, 1, lie))), (br, ('share', (k, 1, r1))), (solver, ('open', k)), (solver, ('submit', task.task.x))]
    if cls is not BridgeBoardEnv:
        b = cls(1800, 16, guardSlots=range(4, 8), squads=4, **kw, **x)
        assert b.tasks[k].task == task.task and b.squadOfTask[k] == s and br % 4 == s
    seq = list(enumerate(mirror + moves, 1))[0 if cls is BridgeBoardEnv else len(mirror):]   # the same times on both boards
    rews = [b.apply(i, a, t)[0] for t, (i, a) in seq]
    return b, k, task.value, (br, h, liar, solver), rews[-len(moves):]

b, k, v, (br, h, liar, solver), rews = play('credit')
assert [x[1:5] for x in b.relayLog] == [(br, k, 1, b.mirrorLog[0][4]), (br, k, 1, b.tasks[k].task.clue(1)[1])]
assert b.relayLog[0][5] == b.mirrorLog[0] and b.relayLog[1][5] == b.mirrorLog[1]
assert relaySource(b, b.shareLog[0])[1] == liar and relaySource(b, b.shareLog[1])[1] == h   # the liar's lie, relayed
assert relaySource(b, b.relayLog[0]) == b.mirrorLog[0] and relaySource(ColonyBoardEnv(1, 8, **kw), b.shareLog[0]) is None
assert b.shareLog[0][6] and not b.shareLog[1][6] and b.counts['solved'] == 1 and b.tasks[k].solvedBy == solver
assert b._creditTo(solver, k) == [(br, 2), (h, 2)]
assert rews[-1] == 0.5 * v and b.pending[br] == 0.25 * v and b.pending[h] == 0.25 * v
c, *_ = play('contest')
assert c.pending[br] == 0.25 * v and c.pending[h] == 0.25 * v
# without the bridge board (ColonyBoardEnv: the relays are plain shares) the bridge takes the whole credit share
cb, *_ = play('credit', ColonyBoardEnv)
assert cb.pending[br] == (0.5 * v if solver not in cb.tasks[k].holders[1] else 0) and cb.pending[h] == 0
# 'individual': every reward equals ColonyBoardEnv's; mirror shares earn nothing
ib, *_, irews = play('individual')
_, *_, cirews = play('individual', ColonyBoardEnv)
assert irews == cirews == [0.0, 0.0, 0.0, v] and not ib.pending.any()
# a non-bridge's share of a mirror value is no relay (its validity is parse's: T2)
b2 = BridgeBoardEnv(1800, 16, guardSlots=range(4, 8), squads=4, nLiar=4, couple=0.25, bridges=1, **kw)
k2 = next(k for k in b2.open if k in b2.coupled)
s, s2 = b2.coupled[k2]
b2.apply(min(b2.tasks[k2].holders[1]), ('share', (k2, 1, b2.tasks[k2].task.clue(1)[1])), 1)
other = next(i for i in range(8, 16) if i % 4 == s and i not in b2.bridgeSet)
b2.apply(other, ('share', (k2, 1, b2.tasks[k2].task.clue(1)[1])), 2)
assert len(b2.shareLog) == 1 and b2.relayLog == [] and relaySource(b2, b2.shareLog[0]) is None

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()

In [ ]:
# couple guard (spec review, 2026-11-10): a bridge key without `couple` is refused, never coupled at the 0.25 default
from explRL.clue_colony import checkOptions
from explRL.colony_grid import WORLD18
for w in (dict(squads=8, bridges=0), dict(squads=8, bridges=1), dict(squads=8, bridgeGuard=True)):
    try:
        checkOptions(dict(WORLD18, colony=True, guards=16, **w), 64, 16); raise AssertionError(w)
    except ValueError:
        pass
assert checkOptions(dict(WORLD18, colony=True, guards=16, squads=8, couple=0.25, bridges=1), 64, 16)
